# ALPR — Colab Runner
**One-time setup:** Run cells 1–4 once per session.
**On code updates:** Just run Cell 2 (`git pull`) then Cell 5 to restart Streamlit.

> Make sure you have selected **Runtime → Change runtime type → T4 GPU** before starting.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# ── Cell 2: Clone or pull latest code from GitHub ─────────────────
# First time:  clones the repo
# Every update: pulls latest changes

import os

REPO_URL = 'https://github.com/YOUR_USERNAME/ALPR.git'  # <-- change this
REPO_DIR = '/content/ALPR'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

print('Done — code is up to date')

In [ ]:
# ── Cell 3: Install dependencies (skip if already installed this session) ──
!pip install -q -r /content/ALPR/requirements.txt
print('Dependencies installed')

In [ ]:
# ── Cell 4: Install + configure ngrok tunnel ───────────────────────
# Get a free ngrok token from https://dashboard.ngrok.com/get-started/your-authtoken

!pip install -q pyngrok

NGROK_TOKEN = 'PASTE_YOUR_NGROK_TOKEN_HERE'  # <-- change this

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
print('ngrok configured')

In [ ]:
# ── Cell 5: Launch Streamlit (re-run this after every git pull) ────
import subprocess, time, urllib.request, urllib.error
from pyngrok import ngrok

# Kill any previous Streamlit process
!pkill -f streamlit || true
time.sleep(2)

# Close any existing ngrok tunnels
ngrok.kill()
time.sleep(1)

# Start Streamlit in background — log output to file so we can debug if needed
log = open('/content/streamlit.log', 'w')
proc = subprocess.Popen(
    ['streamlit', 'run', '/content/ALPR/video_app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=log, stderr=log
)

# Wait until Streamlit is actually accepting connections (up to 60s)
print('Waiting for Streamlit to start', end='')
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:8501', timeout=1)
        print(' ✅ ready')
        break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(1)
else:
    print('\n❌ Streamlit did not start — check /content/streamlit.log')
    !tail -30 /content/streamlit.log
    raise RuntimeError('Streamlit failed to start')

# Open ngrok tunnel only after Streamlit is confirmed running
tunnel = ngrok.connect(8501)
print('=' * 60)
print('  Your app is live at:')
print(f'  {tunnel.public_url}')
print('=' * 60)
print('To update code: git pull in Cell 2, then re-run this cell')

In [ ]:
# ── Cell 6 (optional): Upload a video to Colab for testing ────────
from google.colab import files
uploaded = files.upload()   # pick a video from your computer
for fname in uploaded:
    print(f'Uploaded: /content/{fname}')
    print('Use this path in the video app sidebar')

In [ ]:
# ── Cell 7 (optional): Mount Google Drive ─────────────────────────
# If your videos are already on Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted — files are at /content/drive/MyDrive/')